# Tune `topk_rf`

Hub preprocess (RF top-k + T² + hub×hub interactions) + Random Forest. Repeated stratified CV on the train split; writes [`data/processed/tuned/topk_rf.json`](../data/processed/tuned/topk_rf.json).

Tune all models: [`tune_all.ipynb`](tune_all.ipynb).

**Stage 1 (hyperparameters):** `top_k`, `n_hubs`, classifier `max_depth`. Select by **max mean PR AUC**.

**Stage 2 (threshold):** sweep `THRESHOLD_GRID` on the same CV folds; select threshold that **minimizes mean BER**.

Shared sensor branch adds an **isolation forest** `decision_function` score after T² and hub interactions (unsupervised, refit per CV fold). Grid includes `isolation_forest__n_estimators`.

After hubs: **neighbor_fail_rate** (RF-weighted kNN mean train-label rate; tune `neighbor_fail_rate__n_neighbors`), then isolation forest.


In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd

_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if _cwd.name == "tuning" else _cwd
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import scripts.tuning.registry as tuning_registry

importlib.reload(tuning_registry)
from scripts.secom_pipelines import TARGET_COL, feature_columns, load_mart, split_train_test
from scripts.tuning.registry import (
    MODEL_SPECS,
    fit_with_progress,
    run_grid_search,
    save_tuned_params,
    summarize_cv_search,
    tune_classifier_threshold,
    tuned_params_path,
)

MODEL_ID = "topk_rf"
spec = MODEL_SPECS[MODEL_ID]


In [ ]:
df = load_mart()
feature_cols = feature_columns(df)
train_df, test_df = split_train_test(df)
X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL].astype(int)
print(len(X_train), "train rows", len(test_df), "test rows (holdout, not used here)")


In [ ]:
param_grid = spec.make_param_grid()
pd.DataFrame([{k: v} for k, v in param_grid.items()])


In [ ]:
search, n_candidates, n_splits, total_fits = run_grid_search(spec, X_train, y_train)
print(f"{MODEL_ID}: {n_candidates} candidates x {n_splits} folds = {total_fits} fits")
search = fit_with_progress(search, X_train, y_train)


In [ ]:
cv_summary, fold_results, aggregated = summarize_cv_search(search, spec)
print("Stage 1 best (mean PR AUC):")
display(aggregated.head(10))


In [ ]:
threshold_result = tune_classifier_threshold(spec, X_train, y_train, cv_summary)
print(f"Stage 2 best threshold: {threshold_result['best_threshold']:.2f}")
print(f"  mean BER at threshold: {threshold_result['mean_ber_percent']:.2f}%")
display(threshold_result["per_threshold_mean_ber"].head(10))


In [ ]:
payload = save_tuned_params(
    spec,
    cv_summary,
    fold_results,
    aggregated,
    threshold_result=threshold_result,
)
out_path = tuned_params_path(MODEL_ID)
print(f"Wrote {out_path}")
payload["grid_search_best_params"]
